# Formula 1 Podium Prediction — Milestone 1

**Task:** for every driver in every race, predict whether they finish on the podium (`positionOrder <= 3`)
using only information known **after qualifying and before the race starts**.

**Team:** _add names + IDs_

| # | Section | Owner |
|---|---|---|
| 0 | Setup & shared config | |
| 1 | Load raw data | |
| 2 | EDA before cleaning | |
| 3 | Cleaning: sentinels & semantic typing | |
| 4 | Key audit: primary & foreign keys | |
| 5 | Grain: one row per (raceId, driverId) | |
| 6 | EDA after cleaning | |
| 7 | Data-engineering questions Q1–Q3 | |
| 8 | Feature engineering & selection | |
| 9 | Train / validation / test split | |
| 10 | Models (baseline, statistical, shallow NN) | |
| 11 | Experiments: ablation & preprocessing order | |
| 12 | Explainability (SHAP, permutation, LIME) | |
| 13 | Inference function | |
| 14 | Limitations | |

**Golden rule:** a column may only become a feature if it is known before lights out.
Results, points, laps, status, pit stops, lap times and *post-race* standings are leakage.

## 0. Setup & shared config
Everyone uses these constants. Do not redefine them in later sections.

In [ ]:
import sys, subprocess, importlib
# LIME is not always preinstalled on Kaggle (needs Internet ON in notebook settings)
for pkg in ["lime", "shap"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False, capture_output=True)

import os, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")

SEED = 42
np.random.seed(SEED)

TRAIN_END = 2019            # train: season <= 2019
VAL_YEARS = (2020, 2021)    # validation: 2020-2021
TEST_START = 2022           # test: season >= 2022
SENTINEL = r"\N"           # the dataset's placeholder for "missing"

## 1. Load raw data
Raw tables are loaded **as-is** into `raw` and never modified. All cleaning happens on copies in `clean`.

In [ ]:
def find_data_dir():
    # Kaggle mounts inputs under /kaggle/input/<dataset-slug>/ ; also works locally if CSVs sit in ./data
    hits = glob.glob("/kaggle/input/**/circuits.csv", recursive=True) + glob.glob("./data/**/circuits.csv", recursive=True)
    if not hits:
        raise FileNotFoundError("circuits.csv not found - attach the F1 dataset via 'Add Input'")
    return os.path.dirname(hits[0])

DATA_DIR = find_data_dir()
TABLES = ["circuits", "constructors", "constructor_results", "constructor_standings", "drivers",
          "driver_standings", "races", "results", "qualifying", "lap_times", "pit_stops",
          "sprint_results", "seasons", "status"]

# keep_default_na=False so "\N" stays visible as a string in the raw audit
raw = {t: pd.read_csv(os.path.join(DATA_DIR, f"{t}.csv"), keep_default_na=False, dtype=str) for t in TABLES}

pd.DataFrame({t: {"rows": len(df), "cols": df.shape[1]} for t, df in raw.items()}).T

In [ ]:
# Column names per table - confirm they match the schema we planned around
for t, df in raw.items():
    print(f"{t:24s} {list(df.columns)}")

## 2. EDA before cleaning
Goal: show *with code and plots* what is wrong with the raw data, so every cleaning step is justified.

In [ ]:
# Sentinel audit: how many "\N" values does each column contain?
sentinel_counts = []
for t, df in raw.items():
    for col in df.columns:
        n = (df[col] == SENTINEL).sum()
        if n:
            sentinel_counts.append({"table": t, "column": col, "n_sentinel": n, "pct": 100 * n / len(df)})
sentinel_df = pd.DataFrame(sentinel_counts).sort_values("pct", ascending=False)
sentinel_df

In [ ]:
top = sentinel_df.head(25)
plt.figure(figsize=(9, 7))
plt.barh(top["table"] + "." + top["column"], top["pct"])
plt.gca().invert_yaxis()
plt.xlabel("% of rows equal to \\N")
plt.title("Raw data: share of sentinel values per column (top 25)")
plt.tight_layout(); plt.show()

In [ ]:
# Every column is a string in raw form - this is why semantic typing is needed
raw["results"].dtypes.value_counts()

**TODO (EDA owner):** add more before-cleaning plots, e.g.
- races per season and entrants per race (field size over time)
- podium share per season (class imbalance)
- qualifying coverage per season (needs `qualifying` joined to `races`)

## 3. Cleaning: sentinels & semantic typing
Steps, applied to copies:
1. Replace `\N` with `NaN`.
2. Convert ID / numeric columns to numbers.
3. Convert date strings to `datetime`.
4. Convert lap-time / duration strings (`"1:27.264"`, `"21.664"`) to numeric **seconds**.

In [ ]:
def to_seconds(s):
    """'1:27.264' -> 87.264 ; '21.664' -> 21.664 ; '1:02:03.5' -> 3723.5 ; NaN/garbage -> NaN"""
    if pd.isna(s):
        return np.nan
    s = str(s).strip().lstrip("+")
    try:
        parts = [float(p) for p in s.split(":")]
    except ValueError:
        return np.nan
    secs = 0.0
    for p in parts:
        secs = secs * 60 + p
    return secs

assert to_seconds("1:27.264") == 87.264
assert to_seconds("21.664") == 21.664
assert np.isnan(to_seconds(np.nan))

In [ ]:
DATE_COLS = {"races": ["date", "fp1_date", "fp2_date", "fp3_date", "quali_date", "sprint_date"],
             "drivers": ["dob"]}
TIME_COLS = {"qualifying": ["q1", "q2", "q3"],
             "lap_times": ["time"],
             "pit_stops": ["duration"],
             "results": ["fastestLapTime"],
             "sprint_results": ["fastestLapTime"]}
# Text columns that must stay strings
TEXT_COLS = {"circuitRef", "name", "location", "country", "url", "constructorRef", "nationality",
             "driverRef", "code", "forename", "surname", "positionText", "status", "time"}

clean = {}
for t, df in raw.items():
    d = df.copy().replace(SENTINEL, np.nan)
    for col in d.columns:
        if col in DATE_COLS.get(t, []):
            d[col] = pd.to_datetime(d[col], errors="coerce")
        elif col in TIME_COLS.get(t, []):
            d[col + "_s"] = d[col].map(to_seconds)      # keep original text, add numeric seconds
        elif col not in TEXT_COLS and not col.endswith("_time"):
            converted = pd.to_numeric(d[col], errors="coerce")
            # only convert if it doesn't destroy real values
            if converted.notna().sum() == d[col].notna().sum():
                d[col] = converted
    clean[t] = d

# Sanity: raw must be untouched
assert (raw["results"] == SENTINEL).any().any(), "raw was modified!"
clean["results"].dtypes

In [ ]:
# Check conversions did not silently lose values (non-null before vs after)
loss = []
for t in TIME_COLS:
    for col in TIME_COLS[t]:
        if col in clean[t]:
            before, after = clean[t][col].notna().sum(), clean[t][col + "_s"].notna().sum()
            loss.append({"table": t, "column": col, "non_null_text": before, "non_null_seconds": after, "lost": before - after})
pd.DataFrame(loss)

## 4. Key audit: primary & foreign keys

In [ ]:
PRIMARY_KEYS = {
    "circuits": ["circuitId"], "constructors": ["constructorId"], "drivers": ["driverId"],
    "races": ["raceId"], "seasons": ["year"], "status": ["statusId"],
    "results": ["resultId"], "qualifying": ["qualifyId"], "sprint_results": ["resultId"],
    "driver_standings": ["driverStandingsId"], "constructor_standings": ["constructorStandingsId"],
    "constructor_results": ["constructorResultsId"],
    "lap_times": ["raceId", "driverId", "lap"], "pit_stops": ["raceId", "driverId", "stop"],
}
pk_report = []
for t, keys in PRIMARY_KEYS.items():
    d = clean[t]
    pk_report.append({"table": t, "key": "+".join(keys),
                      "null_rows": int(d[keys].isna().any(axis=1).sum()),
                      "duplicate_rows": int(d.duplicated(keys).sum())})
pd.DataFrame(pk_report)

In [ ]:
# (child table, child column, parent table, parent column)
FOREIGN_KEYS = [("races", "year", "seasons", "year"), ("races", "circuitId", "circuits", "circuitId")]
for child in ["results", "qualifying", "sprint_results", "driver_standings", "lap_times", "pit_stops"]:
    FOREIGN_KEYS += [(child, "raceId", "races", "raceId"), (child, "driverId", "drivers", "driverId")]
for child in ["results", "qualifying", "sprint_results", "constructor_standings", "constructor_results"]:
    FOREIGN_KEYS += [(child, "constructorId", "constructors", "constructorId")]
FOREIGN_KEYS += [("constructor_standings", "raceId", "races", "raceId"),
                 ("constructor_results", "raceId", "races", "raceId"),
                 ("results", "statusId", "status", "statusId"),
                 ("sprint_results", "statusId", "status", "statusId")]

fk_report = []
for ct, cc, pt, pc in FOREIGN_KEYS:
    vals = clean[ct][cc].dropna()
    orphans = ~vals.isin(clean[pt][pc])
    fk_report.append({"child": f"{ct}.{cc}", "parent": f"{pt}.{pc}",
                      "null_fk": int(clean[ct][cc].isna().sum()), "orphans": int(orphans.sum())})
fk_df = pd.DataFrame(fk_report)
fk_df

**TODO:** interpret the tables above. If any orphans or duplicates appear, show examples and decide how to handle them.

## 5. Grain: one row per (raceId, driverId)
The modeling table needs exactly one row per driver per race. Shared drives (mainly the 1950s) break this.

In [ ]:
res = clean["results"]
dup_mask = res.duplicated(["raceId", "driverId"], keep=False)
dups = res[dup_mask].merge(clean["races"][["raceId", "year", "name"]], on="raceId")
print(f"{dup_mask.sum()} rows involved in {res[dup_mask].groupby(['raceId','driverId']).ngroups} duplicated (raceId, driverId) pairs")
dups.groupby("year").size().plot(kind="bar", figsize=(10, 3), title="Duplicated driver-race rows per season")
plt.ylabel("rows"); plt.show()
dups.sort_values(["raceId", "driverId"]).head(10)

In [ ]:
# PROPOSED RULE (team to confirm + justify in markdown):
# keep the car the driver STARTED in (grid > 0); if still tied, keep the best positionOrder.
def dedupe_results(df):
    d = df.copy()
    d["_started"] = (d["grid"].fillna(0) > 0).astype(int)
    d = d.sort_values(["raceId", "driverId", "_started", "positionOrder"], ascending=[True, True, False, True])
    return d.drop_duplicates(["raceId", "driverId"], keep="first").drop(columns="_started")

results_dedup = dedupe_results(res)
assert not results_dedup.duplicated(["raceId", "driverId"]).any()
print(f"rows before: {len(res)}, after: {len(results_dedup)}, removed: {len(res) - len(results_dedup)}")

## 6. EDA after cleaning
**TODO:** repeat the key before-cleaning plots and show what changed.

## 7. Data-engineering questions
Each answer needs: result table, plot, and a note on grain, missingness and the pre-race cutoff.

### Q1. Which circuits convert a front-row start into a podium most often, before vs after 2014?
Tables: `results` + `qualifying` + `races` + `circuits`. Compare qualifying P1–2 vs grid P1–2.

### Q2. Home-race podium effect, controlling for grid position
Tables: `results` + `drivers` + `races` + `circuits`. Needs a nationality → country mapping.

### Q3. Mechanical-retirement rate per constructor, 2014–2021 vs 2022–2024
Tables: `results` + `status` + `constructors` + `races`. Needs a documented status → mechanical mapping.

In [ ]:
# Starting point for Q3: list every status with its frequency so the team can label it
status_freq = (clean["results"].merge(clean["status"], on="statusId")
               .groupby("status").size().sort_values(ascending=False).rename("count"))
status_freq.head(40)

## 8. Feature engineering & selection
Every feature must be (a) known before the race, (b) justified with a plot or statistic.

Candidate groups: **grid**, **qualifying** (position, gap to pole), **driver form** (past races only),
**constructor form** (past races only), **standings before this race** (shifted by one race).

## 9. Train / validation / test split
Time-based: train ≤ 2019, validation 2020–2021, test ≥ 2022.

## 10. Models
Baseline → ≥ 2 statistical models → shallow NN. Report ROC-AUC, PR-AUC, F1 on train/val/test. Thresholds chosen on validation only.

## 11. Experiments
- Feature-group ablation
- ≥ 2 preprocessing-order experiments (e.g. scaler fit on train vs all; impute before vs after split)

## 12. Explainability
Global: permutation importance + SHAP. Local: LIME (+ SHAP waterfall). **Importance ≠ causation.**

## 13. Inference function
Demo on a complete record and a partially-missing record.

## 14. Limitations
- Qualifying-coverage drift
- Era sensitivity
- …